In [3]:
#loading the dataset, it has 10,000 files so they each have to be found recursively, we then append everything and convert it all to a csv 
#so we dont have to load the dataset every single time

import os
import glob
import h5py
import pandas as pd
from tqdm import tqdm

DATA_DIR = "data/MillionSongSubset"

# Recursively find all h5 files across the A/B/C/... subfolders
h5_files = glob.glob(os.path.join(DATA_DIR, "**", "*.h5"), recursive=True)
print(f"Found {len(h5_files)} song files.")

records = []

for filepath in tqdm(h5_files):
    with h5py.File(filepath, "r") as f:
        # Metadata group (songs table)
        meta = f["metadata"]["songs"][0]
        # Analysis group (songs table)
        analysis = f["analysis"]["songs"][0]
        
        records.append({
            "song_id": meta["song_id"].decode("utf-8"),
            "title": meta["title"].decode("utf-8"),
            "artist_name": meta["artist_name"].decode("utf-8"),
            "artist_familiarity": meta["artist_familiarity"],
            "artist_hotttnesss": meta["artist_hotttnesss"],
            "song_hotttnesss": meta["song_hotttnesss"],
            "duration": analysis["duration"],
            "tempo": analysis["tempo"],
            "loudness": analysis["loudness"],
            "key": analysis["key"],
            "mode": analysis["mode"],
            "time_signature": analysis["time_signature"],
        })

df = pd.DataFrame(records)
df.head()



Found 10000 song files.


100%|██████████| 10000/10000 [01:43<00:00, 96.34it/s]


,song_id,title,artist_name,artist_familiarity,artist_hotttnesss,song_hotttnesss,duration,tempo,loudness,key,mode,time_signature
0,SOMZWCG12A8C13C480,I Didn't Mean To,Casual,0.581794,0.401998,0.602120,218.93179,92.198,-11.197,1,0,4
1,SOCIWDW12A8C13D406,Soul Deep,The Box Tops,0.630630,0.417500,NaN,148.03546,121.274,-9.843,6,0,4
2,SOXVLOJ12AB0189215,Amor De Cabaret,Sonora Santanera,0.487357,0.343428,NaN,177.47546,100.070,-9.689,8,1,1
3,SONHOTT12A8C13493C,Something Girls,Adam Ant,0.630382,0.454231,NaN,233.40363,119.293,-9.013,0,1,4
4,SOFSOCN12A8C143F5D,Face the Ashes,Gob,0.651046,0.401724,0.604501,209.60608,129.738,-4.501,2,1,4


In [5]:
#convert to csv
df.to_csv("data/msd_subset_clean.csv", index=False)

In [9]:
#load dataset
# df = pd.read_csv("data/msd_subset_clean.csv")
df

,song_id,title,artist_name,artist_familiarity,artist_hotttnesss,song_hotttnesss,duration,tempo,loudness,key,mode,time_signature
0,SOMZWCG12A8C13C480,I Didn't Mean To,Casual,0.581794,0.401998,0.602120,218.93179,92.198,-11.197,1,0,4
1,SOCIWDW12A8C13D406,Soul Deep,The Box Tops,0.630630,0.417500,NaN,148.03546,121.274,-9.843,6,0,4
2,SOXVLOJ12AB0189215,Amor De Cabaret,Sonora Santanera,0.487357,0.343428,NaN,177.47546,100.070,-9.689,8,1,1
3,SONHOTT12A8C13493C,Something Girls,Adam Ant,0.630382,0.454231,NaN,233.40363,119.293,-9.013,0,1,4
4,SOFSOCN12A8C143F5D,Face the Ashes,Gob,0.651046,0.401724,0.604501,209.60608,129.738,-4.501,2,1,4
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,SOLXXPY12A67ADABA0,The Hanged Man,Moonspell,0.722571,0.499826,0.594080,386.19383,140.185,-8.087,7,1,4
9996,SOAYONI12A6D4F85C8,The Wonderful World Of The Young,Danny Williams,0.511663,0.409779,0.334707,168.01914,77.072,-14.517,8,1,3
9997,SOJZLAJ12AB017E8A2,Sentimental Man,Winston Reedy,0.433508,0.289903,NaN,193.72363,118.123,-12.087,1,1,4
9998,SORZSCJ12A8C132446,Zydeco In D-Minor,"Myrick ""Freeze"" Guillory",0.334457,0.216829,0.000000,300.82567,137.663,-12.574,0,1,4


In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

#we were curious whether or not songs in major or minor have huge differences in tempo, loudness, duration, key signature, and time signature.
#we get inputs and targets, split our data 80/20 into train and test sets, and test on a dummy to get a baseline accuracy test.
#after this, we test while trying euclidean and manhattan distance (since we learned these two in class) and then print the results

feature_cols = ["duration", "tempo", "loudness", "key", "time_signature"]

df_clean = df.dropna(subset=feature_cols + ["mode"]).copy()

x = df_clean[feature_cols]
y = df_clean["mode"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.20, random_state=42, stratify=y
)

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(x_train, y_train)
dummy_preds = dummy.predict(x_test)
print(f"Baseline Balanced Accuracy: {balanced_accuracy_score(y_test, dummy_preds):.3f}\n")

for k in [1, 3, 5, 11, 21, 35, 51, 75, 101]:
    for distance_metric in ["euclidean", "manhattan"]:
        model_knn = Pipeline([
            ("scaler", StandardScaler()),
            ("knn", KNeighborsClassifier(n_neighbors=k, metric=distance_metric, weights="distance"))
        ])
        model_knn.fit(x_train, y_train)
        preds = model_knn.predict(x_test)
        probs = model_knn.predict_proba(x_test)[:, 1]
        
        accuracy_score = balanced_accuracy_score(y_test, preds)
        auc = roc_auc_score(y_test, probs)
        print(f"k={k:<2} | metric={distance_metric:<9} | Balanced Acc: {accuracy_score:.3f} | ROC-AUC: {auc:.3f}")

Baseline Balanced Accuracy: 0.500

k=1  | metric=euclidean | Balanced Acc: 0.527 | ROC-AUC: 0.527
k=1  | metric=manhattan | Balanced Acc: 0.543 | ROC-AUC: 0.543
k=3  | metric=euclidean | Balanced Acc: 0.534 | ROC-AUC: 0.555
k=3  | metric=manhattan | Balanced Acc: 0.548 | ROC-AUC: 0.572
k=5  | metric=euclidean | Balanced Acc: 0.542 | ROC-AUC: 0.578
k=5  | metric=manhattan | Balanced Acc: 0.553 | ROC-AUC: 0.591
k=11 | metric=euclidean | Balanced Acc: 0.552 | ROC-AUC: 0.605
k=11 | metric=manhattan | Balanced Acc: 0.562 | ROC-AUC: 0.605
k=21 | metric=euclidean | Balanced Acc: 0.551 | ROC-AUC: 0.618
k=21 | metric=manhattan | Balanced Acc: 0.564 | ROC-AUC: 0.627
k=35 | metric=euclidean | Balanced Acc: 0.559 | ROC-AUC: 0.626
k=35 | metric=manhattan | Balanced Acc: 0.561 | ROC-AUC: 0.634
k=51 | metric=euclidean | Balanced Acc: 0.559 | ROC-AUC: 0.631
k=51 | metric=manhattan | Balanced Acc: 0.556 | ROC-AUC: 0.641
k=75 | metric=euclidean | Balanced Acc: 0.553 | ROC-AUC: 0.628
k=75 | metric=manhat